# Module 03: Object-Oriented Programming for ML Projects

Learn how to design reusable, production-ready ML components using OOP patterns.

**Focus areas:**
- Classes, methods, `self`, `__init__`
- Inheritance, polymorphism
- Properties, static/class methods
- Magic methods, dataclasses, ABCs
- ML Model, Dataset, and Pipeline class design

## 1. Classes: The Blueprint

A class bundles data (attributes) and behavior (methods) into one unit.

**ML Analogy:** A `LinearRegression` class contains both the coefficients (data) and the `predict()` logic (behavior).

In [ ]:
class Model:
    """Simple ML model class."""
    
    def __init__(self, name, learning_rate=0.01):
        """Initialize model with name and hyperparameters."""
        self.name = name
        self.learning_rate = learning_rate
        self.weights = None
        self._is_trained = False
    
    def train(self, X, y):
        """Train the model on data X with labels y."""
        print("Training", self.name, "with lr =", self.learning_rate)
        self.weights = [0.5] * len(X[0]) if X else []
        self._is_trained = True
        return self
    
    def predict(self, X):
        """Make predictions. Raises error if not trained."""
        if not self._is_trained:
            raise ValueError("Model not trained. Call train() first.")
        return [sum(w * x[j] for j, w in enumerate(self.weights)) for x in X]


# Create and use a model
model = Model("LinearRegressor", 0.01)
X = [[1.0, 2.0], [3.0, 4.0]]
y = [3.0, 7.0]
model.train(X, y)
preds = model.predict([[2.0, 3.0]])
print("Predictions:", preds)
print("Weights:", model.weights)
print("Name:", model.name)

## 2. Inheritance: Building on Existing Code

Create a hierarchy of models that share the `fit()/predict()` interface but implement them differently.

In [ ]:
class BaseEstimator:
    """Base class for all ML estimators."""
    
    def __init__(self, name="model"):
        self.name = name
    
    def fit(self, X, y):
        """Fit model to data. Must be overridden by subclasses."""
        raise NotImplementedError("Subclasses must implement fit()")
    
    def predict(self, X):
        """Predict on new data. Must be overridden by subclasses."""
        raise NotImplementedError("Subclasses must implement predict()")


class LogisticRegression(BaseEstimator):
    """Logistic Regression classifier."""
    
    def __init__(self, lr=0.01, max_iter=100):
        super().__init__("LogisticRegression")
        self.lr = lr
        self.max_iter = max_iter
        self.coef_ = None
    
    def fit(self, X, y):
        print(self.name, "fitting with lr =", self.lr)
        self.coef_ = [0.1] * len(X[0])
        return self
    
    def predict(self, X):
        return [1 if sum(w * x[i] for i, w in enumerate(self.coef_)) > 0 else 0 for x in X]


class RandomForest(BaseEstimator):
    """Random Forest classifier."""
    
    def __init__(self, n_trees=10, max_depth=5):
        super().__init__("RandomForest")
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []
    
    def fit(self, X, y):
        print(self.name, "fitting with", self.n_trees, "trees")
        self.trees = ["tree_" + str(i) for i in range(self.n_trees)]
        return self
    
    def predict(self, X):
        return [0] * len(X)


# Polymorphism in action
models = [LogisticRegression(), RandomForest()]
X_train = [[1, 2], [3, 4], [5, 6]]
y_train = [0, 1, 0]
for m in models:
    m.fit(X_train, y_train)
    print(m.name, "predicts:", m.predict([[1, 1]]))

## 3. Properties: Computed Attributes

Use `@property` to create attributes computed on the fly. No manual syncing needed.

In [ ]:
class Dataset:
    """Encapsulates a dataset with metadata."""
    
    def __init__(self, features, labels=None, name="dataset"):
        self._features = features
        self._labels = labels
        self.name = name
    
    @property
    def num_samples(self):
        """Number of samples (rows)."""
        return len(self._features)
    
    @property
    def num_features(self):
        """Number of features (columns)."""
        if not self._features:
            return 0
        return len(self._features[0])
    
    @property
    def shape(self):
        """Tuple of (samples, features)."""
        return (self.num_samples, self.num_features)
    
    @property
    def has_labels(self):
        """Whether labels are available."""
        return self._labels is not None
    
    @property
    def class_distribution(self):
        """Count of each unique label."""
        if not self.has_labels:
            return None
        counts = {}
        for label in self._labels:
            counts[label] = counts.get(label, 0) + 1
        return counts


# Example usage
data = Dataset([[1, 2], [3, 4], [5, 6]], [0, 1, 0], "training")
print("Dataset:", data.name)
print("Shape:", data.shape)
print("Samples:", data.num_samples, "Features:", data.num_features)
print("Has labels:", data.has_labels)
print("Class distribution:", data.class_distribution)

## 4. @staticmethod and @classmethod

- `@staticmethod`: Utility function that belongs logically to the class
- `@classmethod`: Alternative constructors or factory methods

In [ ]:
class DataLoader:
    """Handles data loading from various sources."""
    
    supported_formats = ["csv", "json", "parquet"]
    
    def __init__(self, filepath):
        self.filepath = filepath
    
    @classmethod
    def from_csv(cls, filepath):
        """Factory: create a DataLoader from a CSV path."""
        print("Creating CSV loader for:", filepath)
        return cls(filepath)
    
    @classmethod
    def from_s3(cls, bucket, key):
        """Factory: download from S3 then create loader."""
        filepath = "/local/" + key.split("/")[-1]
        print("Downloading from s3://" + bucket + "/" + key)
        print("Saved to:", filepath)
        return cls(filepath)
    
    @staticmethod
    def is_supported(format_name):
        """Check if a format is supported."""
        return format_name.lower() in DataLoader.supported_formats
    
    def load(self):
        """Load data from filepath."""
        print("Loading data from:", self.filepath)
        return [[1, 2, 3], [4, 5, 6]]  # dummy data


# Using class methods as factories
loader1 = DataLoader.from_csv("train.csv")
loader2 = DataLoader.from_s3("my-bucket", "data/train.parquet")

# Using static method
print("CSV supported?", DataLoader.is_supported("csv"))
print("Excel supported?", DataLoader.is_supported("xlsx"))

# Load data
data = loader1.load()
print("Loaded data:", data)

## 5. Magic (Dunder) Methods

Magic methods let custom objects work with Python's built-in functions and operators.

In [ ]:
class FeatureMatrix:
    """A 2D feature matrix with numpy-like indexing."""
    
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        """Return number of rows."""
        return len(self.data)
    
    def __getitem__(self, idx):
        """Support indexing: fm[0], fm[1:3], fm[0, 1]."""
        if isinstance(idx, tuple):
            row, col = idx
            return self.data[row][col]
        return self.data[idx]
    
    def __setitem__(self, idx, value):
        """Support assignment: fm[0] = [1, 2, 3]."""
        self.data[idx] = value
    
    def __str__(self):
        """Human-readable string representation."""
        return "FeatureMatrix(" + str(len(self)) + " rows x " + str(len(self.data[0])) + " cols)"
    
    def __repr__(self):
        """Detailed representation for debugging."""
        return "FeatureMatrix(" + repr(self.data) + ")"
    
    def __add__(self, other):
        """Element-wise addition of two matrices."""
        result = []
        for i in range(len(self.data)):
            row = []
            for j in range(len(self.data[0])):
                row.append(self.data[i][j] + other.data[i][j])
            result.append(row)
        return FeatureMatrix(result)


fm1 = FeatureMatrix([[1, 2], [3, 4]])
fm2 = FeatureMatrix([[5, 6], [7, 8]])

print("fm1:", fm1)
print("len(fm1):", len(fm1))
print("fm1[0]:", fm1[0])
print("fm1[0, 1]:", fm1[0, 1])

fm3 = fm1 + fm2
print("fm1 + fm2:", fm3)
print("repr:", repr(fm3))

## 6. Dataclasses for Config Objects

Dataclasses auto-generate `__init__`, `__repr__`, `__eq__`, and `__hash__`.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class ModelConfig:
    """Configuration for model training."""
    learning_rate: float = 0.001
    batch_size: int = 32
    num_epochs: int = 10
    hidden_layers: List[int] = field(default_factory=lambda: [64, 32])
    activation: str = "relu"
    random_state: int = 42
    model_name: str = "default_model"
    
    @property
    def total_params_estimate(self):
        """Estimate total parameters from layer sizes."""
        total = 0
        layers = self.hidden_layers
        for i in range(len(layers)):
            in_size = layers[i - 1] if i > 0 else layers[0]
            out_size = layers[i]
            total += (in_size * out_size) + out_size
        return total


@dataclass
class ExperimentResult:
    """Stores results from a single experiment."""
    model_name: str
    accuracy: float
    precision: float
    recall: float
    f1_score: float
    train_time_seconds: float
    config: ModelConfig


# Usage
config1 = ModelConfig(learning_rate=0.01, num_epochs=20)
config2 = ModelConfig()
print("Config 1:", config1)
print("Config 2:", config2)
print("Configs equal?", config1 == config2)
print("Total params estimate:", config1.total_params_estimate)

result = ExperimentResult(
    model_name="test_model",
    accuracy=0.95,
    precision=0.93,
    recall=0.94,
    f1_score=0.935,
    train_time_seconds=120.5,
    config=config1
)
print("\nExperiment result:", result)

## 7. Abstract Base Classes (ABC) for Interfaces

ABCs define a contract. Any class inheriting from an ABC must implement all abstract methods.

In [ ]:
from abc import ABC, abstractmethod


class BaseTransformer(ABC):
    """Abstract base class for all data transformers."""
    
    @abstractmethod
    def fit(self, X):
        """Learn parameters from data."""
        pass
    
    @abstractmethod
    def transform(self, X):
        """Apply transformation to data."""
        pass
    
    def fit_transform(self, X):
        """Fit and transform in one step."""
        self.fit(X)
        return self.transform(X)


class StandardScaler(BaseTransformer):
    """Standardize features by removing mean and scaling to unit variance."""
    
    def __init__(self):
        self.mean_ = None
        self.std_ = None
    
    def fit(self, X):
        n = len(X)
        n_features = len(X[0])
        self.mean_ = [0.0] * n_features
        self.std_ = [0.0] * n_features
        
        for j in range(n_features):
            col_sum = sum(X[i][j] for i in range(n))
            self.mean_[j] = col_sum / n
        
        for j in range(n_features):
            var = sum((X[i][j] - self.mean_[j]) ** 2 for i in range(n)) / n
            self.std_[j] = var ** 0.5
        
        return self
    
    def transform(self, X):
        result = []
        for x in X:
            transformed = [(x[j] - self.mean_[j]) / self.std_[j] if self.std_[j] > 0 else 0.0 for j in range(len(x))]
            result.append(transformed)
        return result


class MinMaxScaler(BaseTransformer):
    """Scale features to a [0, 1] range."""
    
    def __init__(self):
        self.min_ = None
        self.max_ = None
    
    def fit(self, X):
        n_features = len(X[0])
        self.min_ = [float('inf')] * n_features
        self.max_ = [float('-inf')] * n_features
        for x in X:
            for j in range(n_features):
                if x[j] < self.min_[j]:
                    self.min_[j] = x[j]
                if x[j] > self.max_[j]:
                    self.max_[j] = x[j]
        return self
    
    def transform(self, X):
        result = []
        for x in X:
            scaled = [(x[j] - self.min_[j]) / (self.max_[j] - self.min_[j]) if self.max_[j] > self.min_[j] else 0.0 for j in range(len(x))]
            result.append(scaled)
        return result


# Polymorphic transformer usage
X = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]

transformers = [StandardScaler(), MinMaxScaler()]
for t in transformers:
    t.fit(X)
    X_transformed = t.transform(X)
    print(t.__class__.__name__ + ":", X_transformed)

## 8. Building an ML Pipeline with Composition

A Pipeline *has* transformers and a model. This composition pattern is better than deep inheritance.

In [ ]:
class Pipeline:
    """Chain multiple transformers and a final estimator."""
    
    def __init__(self, steps):
        """Initialize pipeline.
        
        Args:
            steps: List of (name, transformer_or_estimator) tuples
        """
        self.steps = steps
    
    def fit(self, X, y=None):
        """Fit all steps in sequence."""
        X_current = X
        for name, step in self.steps[:-1]:
            print("Fitting transformer:", name)
            step.fit(X_current)
            X_current = step.transform(X_current)
        
        name, model = self.steps[-1]
        print("Fitting model:", name)
        model.fit(X_current, y)
        return self
    
    def predict(self, X):
        """Transform X through all steps and predict."""
        X_current = X
        for name, step in self.steps[:-1]:
            print("Transforming with:", name)
            X_current = step.transform(X_current)
        
        name, model = self.steps[-1]
        print("Predicting with:", name)
        return model.predict(X_current)
    
    def __str__(self):
        """String representation of the pipeline."""
        step_names = [name for name, _ in self.steps]
        return "Pipeline(" + " -> ".join(step_names) + ")"


# Build and use a pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(lr=0.01))
])

print("Pipeline:", pipe)
X_train = [[1, 2], [3, 4], [5, 6], [7, 8]]
y_train = [0, 0, 1, 1]

pipe.fit(X_train, y_train)
print("\nPredictions:", pipe.predict([[2, 3], [6, 7]]))

## Summary

| OOP Concept | ML Application |
|---|---|
| Classes | Model, Dataset, Scaler, Pipeline |
| Inheritance | BaseModel -> LogisticRegression, RandomForest |
| Polymorphism | Loop over models and call `.predict()` |
| `@property` | `dataset.shape`, `dataset.class_distribution` |
| `@classmethod` | `DataLoader.from_csv()`, `DataLoader.from_s3()` |
| `@staticmethod` | Utility checks like `is_supported()` |
| Magic methods | `len(dataset)`, `dataset[i]`, `str(pipeline)` |
| `@dataclass` | ModelConfig, ExperimentResult |
| ABC | `BaseTransformer` interface contract |
| Composition | `Pipeline` contains transformers + model |

**Key takeaway:** OOP is not about "fancy code." It's about organizing complexity so your ML projects scale from notebooks to production.